In [1]:
from __future__ import annotations

import argparse
import ast
import json
import sys
import uuid
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Tuple

import numpy as np
import pandas as pd
from tools.mcp_managers.client_manager import MCPClientManager

In [2]:
config_path = "/hpc2hdd/home/zwang374/verl/tools/mcp_configs/bfcl_mcp_server.json"

manager = MCPClientManager()
manager.init_config(config_path)


In [3]:
manager.tools

{'file_system-cat': {'type': 'function',
  'function': {'name': 'file_system-cat',
   'description': '\n        Display the contents of a file of any extension from currrent directory.\n\n        Args:\n            file_name (str): The name of the file from current directory to display. No path is allowed.\n\n        Returns:\n            file_content (str): The content of the file.\n        ',
   'parameters': {'properties': {'file_name': {'title': 'File Name',
      'type': 'string'}},
    'required': ['file_name'],
    'title': 'catArguments',
    'type': 'object'}}},
 'file_system-cd': {'type': 'function',
  'function': {'name': 'file_system-cd',
   'description': '\n        Change the current working directory to the specified folder.\n\n        Args:\n            folder (str): The folder of the directory to change to. You can only change one folder at a time.\n\n        Returns:\n            current_working_directory (str): The new current working directory path.\n        ',
   '

In [4]:
client_id = 'file_system-test-12345'

scenario = {"file_system": {
                "root": {
                    "workspace": {
                        "type": "directory",
                        "contents": {
                            "document": {
                                "type": "directory",
                                "contents": {
                                    "final_report.pdf": {
                                        "type": "file",
                                        "content": "Year2024 This is the final report content including budget analysis and other sections."
                                    },
                                    "previous_report.pdf": {
                                        "type": "file",
                                        "content": "Year203 This is the previous report content with different budget analysis."
                                    }
                                }
                            },
                            "archive": {
                                "type": "directory",
                                "contents": {}
                            }
                        }
                    }
                },
                "current_dir": "/workspace"
            }
}


In [5]:
scenario['file_system']

{'root': {'workspace': {'type': 'directory',
   'contents': {'document': {'type': 'directory',
     'contents': {'final_report.pdf': {'type': 'file',
       'content': 'Year2024 This is the final report content including budget analysis and other sections.'},
      'previous_report.pdf': {'type': 'file',
       'content': 'Year203 This is the previous report content with different budget analysis.'}}},
    'archive': {'type': 'directory', 'contents': {}}}}},
 'current_dir': '/workspace'}

In [6]:
result = manager.call_tool('file_system-load_scenario',scenario['file_system'], client_id)
print(result)


[09/29/25 20:38:37] ERROR    [Client-59e9] Error parsing structured content: make_dataclass() got an  ]8;id=551784;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=430255;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             unexpected keyword argument 'kw_only'                                                 

Load scenario succeeded without checking: Successfully loaded scenario
Successfully loaded scenario


In [7]:

# 尝试调用pwd工具
print('Testing pwd tool...')
try:
    result = manager.call_tool('file_system-pwd', {}, client_id)
    print('pwd result:', result)
except Exception as e:
    print('pwd error:', e)

# 尝试调用ls工具
print('\\nTesting ls tool...')
try:
    result = manager.call_tool('file_system-ls', {}, client_id)
    print('ls result:', result)
except Exception as e:
    print('ls error:', e)

# 关闭client
manager.close_client(client_id)

Testing pwd tool...
file_system-pwd execute: {
  "current_working_directory": "/workspace"
}
pwd result: {
  "current_working_directory": "/workspace"
}
\nTesting ls tool...


[09/29/25 20:38:47] ERROR    [Client-59e9] Error parsing structured content: make_dataclass() got an  ]8;id=879429;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=975484;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             unexpected keyword argument 'kw_only'                                                 

file_system-ls execute: {
  "current_directory_content": [
    "document",
    "archive"
  ]
}
ls result: {
  "current_directory_content": [
    "document",
    "archive"
  ]
}
Client file_system-test-12345 closed and removed


In [8]:
import json
import pandas as pd
from tools.mcp_managers.client_manager import MCPClientManager
import random

# 初始化 client_manager
client_manager = MCPClientManager()
client_manager.init_config('/hpc2hdd/home/zwang374/verl/tools/mcp_configs/bfcl_mcp_server.json')

# 读取 parquet 数据
data = pd.read_parquet('/hpc2hdd/home/zwang374/verl/data/BFCL/multi-turn/train.parquet')

total = 0
success = 0
failure = 0

for i, row in data.iterrows():
    initial_config = json.loads(row['extra_info']['initial_config'])
    involved_classes = row['extra_info']['involved_class']

    for cls_name in involved_classes:
        if cls_name in ['ticket','posting','trading','file_system','travel','message','vehicle']: # 'ticket','posting','trading','file_system','travel','message','vehicle'
            client_id = f"{cls_name}-load_scenario-{random.randint(0, 1000000)}"
            try:    
                scenario = initial_config[cls_name]
                print(scenario)
            except:
                continue
            
            total += 1
            try:
                return_message = client_manager.load_scenario(
                    client_id=client_id,
                    scenario=scenario,
                    check=True  # 打开检查模式
                )
            except Exception:
                failure += 1
            client_manager.close_client(client_id)

print(f"\n=== Summary ===")
print(f"Total attempts: {total}")
print(f"Success: {success}")
print(f"Failure: {failure}")
print(f"Success rate: {success/total:.2%}")


{'username': 'analyst_pro', 'password': 'Kj8#mP9$vL2', 'authenticated': False, 'tweets': {'0': {'id': 0, 'username': 'analyst_pro', 'content': 'Just finished analyzing the reports!', 'tags': ['#analysis', '#reports'], 'mentions': []}, '1': {'id': 1, 'username': 'analyst_pro', 'content': 'Budget analysis insights coming soon!', 'tags': ['#budget', '#analysis', '#insights'], 'mentions': []}, '2': {'id': 2, 'username': 'analyst_pro', 'content': 'Stay tuned for more updates!', 'tags': ['#updates', '#staytuned'], 'mentions': []}}, 'comments': {}, 'retweets': {}, 'following_list': ['alice', 'bob'], 'tweet_counter': 3}


[09/29/25 20:39:04] ERROR    [Client-cf5f] Error parsing structured content: make_dataclass() got an  ]8;id=167469;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=880237;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             unexpected keyword argument 'kw_only'                                                 

Load scenario failed: list index out of range
Client posting-load_scenario-551375 closed and removed
{'root': {'workspace': {'type': 'directory', 'contents': {'document': {'type': 'directory', 'contents': {'final_report.pdf': {'type': 'file', 'content': 'Year2024 This is the final report content including budget analysis and other sections.'}, 'previous_report.pdf': {'type': 'file', 'content': 'Year203 This is the previous report content with different budget analysis.'}}}, 'archive': {'type': 'directory', 'contents': {}}}}}, 'current_dir': '/workspace'}


[09/29/25 20:39:05] ERROR    [Client-391f] Error parsing structured content:                          ]8;id=757848;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=978418;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

save_scenario execute: {
  "root": {
    "workspace": {
      "type": "directory",
      "contents": {
        "document": {
          "type": "directory",
          "contents": {
            "final_report.pdf": {
              "type": "file",
              "content": "Year2024 This is the final report content including budget analysis and other sections."
            },
            "previous_report.pdf": {
              "type": "file",
              "content": "Year203 This is the previous report content with different budget analysis."
            }
          }
        },
        "archive": {
          "type": "directory",
          "contents": {}
        }
      }
    }
  },
  "current_dir": "/workspace"
}
Load scenario succeeded with checking: Successfully loaded scenario
Client file_system-load_scenario-126547 closed and removed
{'username': 'analyst_pro', 'password': 'Kj8#mP9$vL2', 'authenticated': False, 'tweets': {'0': {'id': 0, 'username': 'analyst_pro', 'content': 'Just finis

...

save_scenario fail before execution: Error executing tool save_scenario: 6 validation errors for save_scenarioOutput
result.token_expires_in.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.token_expires_in.str
  Input should be a valid string [type=string_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.budget_limit.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.budget_limit.str
  Input should be a valid string [type=string_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.random_seed.dict[any,any]
  Input should be a valid dictionary [type=dict_ty

[09/29/25 20:51:37] ERROR    [Client-58b1] Error parsing structured content:                          ]8;id=543522;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=632388;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

save_scenario fail before execution: Error executing tool save_scenario: 6 validation errors for save_scenarioOutput
result.token_expires_in.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.token_expires_in.str
  Input should be a valid string [type=string_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.budget_limit.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.budget_limit.str
  Input should be a valid string [type=string_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.random_seed.dict[any,any]
  Input should be a valid dictionary [type=dict_ty

[09/29/25 20:51:38] ERROR    [Client-daf0] Error parsing structured content:                          ]8;id=38098;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=340158;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

save_scenario fail before execution: Error executing tool save_scenario: 6 validation errors for save_scenarioOutput
result.token_expires_in.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.token_expires_in.str
  Input should be a valid string [type=string_type, input_value=3600, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.budget_limit.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/dict_type
result.budget_limit.str
  Input should be a valid string [type=string_type, input_value=5000.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
result.random_seed.dict[any,any]
  Input should be a valid dictionary [type=dict_ty

In [1]:
import json
import pandas as pd
from tools.mcp_managers.client_manager import MCPClientManager
import random

# 初始化 client_manager
client_manager = MCPClientManager()
client_manager.init_config('/hpc2hdd/home/zwang374/verl/tools/mcp_configs/bfcl_mcp_server.json')

# 读取 parquet 数据
data = pd.read_parquet('/hpc2hdd/home/zwang374/verl/data/BFCL/multi-turn/train.parquet')

total = 0
success = 0
failure = 0

for i, row in data.iterrows():
    initial_config = json.loads(row['extra_info']['initial_config'])
    involved_classes = row['extra_info']['involved_class']

    for cls_name in involved_classes:
        if cls_name in ['ticket','posting','trading','file_system','travel','message','vehicle']: # 'ticket','posting','trading','file_system','travel','message','vehicle'
            client_id = f"{cls_name}-load_scenario-{random.randint(0, 1000000)}"
            try:    
                scenario = initial_config[cls_name]
                print(scenario)
            except:
                continue
            
            total += 1
            try:
                return_message = client_manager.load_scenario(
                    client_id=client_id,
                    scenario=scenario,
                    check=True  # 打开检查模式
                )
            except Exception:
                failure += 1
            client_manager.close_client(client_id)

print(f"\n=== Summary ===")
print(f"Total attempts: {total}")
print(f"Success: {success}")
print(f"Failure: {failure}")
print(f"Success rate: {success/total:.2%}")


{'username': 'analyst_pro', 'password': 'Kj8#mP9$vL2', 'authenticated': False, 'tweets': {'0': {'id': 0, 'username': 'analyst_pro', 'content': 'Just finished analyzing the reports!', 'tags': ['#analysis', '#reports'], 'mentions': []}, '1': {'id': 1, 'username': 'analyst_pro', 'content': 'Budget analysis insights coming soon!', 'tags': ['#budget', '#analysis', '#insights'], 'mentions': []}, '2': {'id': 2, 'username': 'analyst_pro', 'content': 'Stay tuned for more updates!', 'tags': ['#updates', '#staytuned'], 'mentions': []}}, 'comments': {}, 'retweets': {}, 'following_list': ['alice', 'bob'], 'tweet_counter': 3}


[09/29/25 19:33:23] ERROR    [Client-a174] Error parsing structured content: make_dataclass() got an  ]8;id=172955;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=443339;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             unexpected keyword argument 'kw_only'                                                 

Load scenario failed: list index out of range
Client posting-load_scenario-819243 closed and removed
{'root': {'workspace': {'type': 'directory', 'contents': {'document': {'type': 'directory', 'contents': {'final_report.pdf': {'type': 'file', 'content': 'Year2024 This is the final report content including budget analysis and other sections.'}, 'previous_report.pdf': {'type': 'file', 'content': 'Year203 This is the previous report content with different budget analysis.'}}}, 'archive': {'type': 'directory', 'contents': {}}}}}, 'current_dir': '/workspace'}


                    ERROR    [Client-83dd] Error parsing structured content:                          ]8;id=268829;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=600393;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

Load scenario failed: list index out of range
Client file_system-load_scenario-690716 closed and removed
{'username': 'analyst_pro', 'password': 'Kj8#mP9$vL2', 'authenticated': False, 'tweets': {'0': {'id': 0, 'username': 'analyst_pro', 'content': 'Just finished analyzing the reports!', 'tags': ['#analysis', '#reports'], 'mentions': []}, '1': {'id': 1, 'username': 'analyst_pro', 'content': 'Budget analysis insights coming soon!', 'tags': ['#budget', '#analysis', '#insights'], 'mentions': []}, '2': {'id': 2, 'username': 'analyst_pro', 'content': 'Stay tuned for more updates!', 'tags': ['#updates', '#staytuned'], 'mentions': []}}, 'comments': {}, 'retweets': {}, 'following_list': ['alice', 'bob'], 'tweet_counter': 3}


...

[09/29/25 19:35:32] ERROR    [Client-8310] Error parsing structured content:                          ]8;id=915439;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=281743;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

Load scenario failed: list index out of range
Client file_system-load_scenario-789123 closed and removed
{'root': {'temp': {'type': 'directory', 'contents': {'file1.txt': {'type': 'file', 'content': 'Line 1\nLine 2\nLine 3\nLine 4\nLine 5\nLine 6\nLine 7\nLine 8\nLine 9\nLine 10\nLine 11\nLine 12\nLine 13\nLine 14\nLine 15\nLine 16\nLine 17\nLine 18\nLine 19\nLine 20'}, 'file2.txt': {'type': 'file', 'content': 'Alpha\nBeta\nGamma\nDelta\nEpsilon\nZeta\nEta\nTheta\nIota\nKappa\nLambda\nMu\nNu\nXi\nOmicron\nPi\nRho\nSigma\nTau\nUpsilon'}, 'file3.txt': {'type': 'file', 'content': 'Zebra\nApple\nOrange\nBanana\nGrape\nCherry\nMango\nPeach\nLemon\nLime\nKiwi\nPlum\nPear\nFig\nDate\nCoconut\nPineapple\nPapaya\nGuava\nLychee'}}}}, 'current_dir': '/temp'}


                    ERROR    [Client-d5d1] Error parsing structured content:                          ]8;id=624715;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py\client.py]8;;\:]8;id=23337;file:///hpc2hdd/home/zwang374/anaconda3/envs/verl/lib/python3.10/site-packages/fastmcp/client/client.py#916\916]8;;\
                             `TypeAdapter[ForwardRef('load_scenarioOutput')]` is not fully defined;                
                             you should define `ForwardRef('load_scenarioOutput')` and all referenced              
                             types, then call `.rebuild()` on the instance.                                        
                                                                                                                   
                             For further information visit                                                         
                             https://errors.pydantic.dev/2.11/u/class-not-fully-defined                            

Load scenario failed: list index out of range
Client file_system-load_scenario-554404 closed and removed
{'root': {'temp': {'type': 'directory', 'contents': {'file1.txt': {'type': 'file', 'content': 'Line 1\nLine 2\nLine 3\nLine 4\nLine 5\nLine 6\nLine 7\nLine 8\nLine 9\nLine 10\nLine 11\nLine 12\nLine 13\nLine 14\nLine 15\nLine 16\nLine 17\nLine 18\nLine 19\nLine 20'}, 'file2.txt': {'type': 'file', 'content': 'Alpha\nBeta\nGamma\nDelta\nEpsilon\nZeta\nEta\nTheta\nIota\nKappa\nLambda\nMu\nNu\nXi\nOmicron\nPi\nRho\nSigma\nTau\nUpsilon'}, 'file3.txt': {'type': 'file', 'content': 'Zebra\nApple\nOrange\nBanana\nGrape\nCherry\nMango\nPeach\nLemon\nLime\nKiwi\nPlum\nPear\nFig\nDate\nCoconut\nPineapple\nPapaya\nGuava\nLychee'}}}}, 'current_dir': '/temp'}


KeyboardInterrupt: 